# MLJAR AutoML

<div style="text-align: center;"><img src=".//Images//AutoML_white.png" alt="AutoML" width="200" height="200" style="margin: 10px; "/></div>

[MLJAR-Supervised](https://github.com/mljar/mljar-supervised) to biblioteka AutoML napisana w Pythonie, przeznaczona do **automatycznej budowy modeli dla problemów klasyfikacji i regresji**. Została zaprojektowana z myślą o prostocie – użytkownik podaje dane, a MLJAR wykonuje wszystkie etapy budowy modelu: od wstępnego przetwarzania danych, przez trenowanie modeli, aż po ocenę i raportowanie wyników.

---

**Zalety:**

* Działa na Windows, Linux i macOS.
* Wspiera klasyfikację i regresję (także dane niezbalansowane).
* Umożliwia działanie w różnych trybach (od szybkiego testowania do pełnej optymalizacji).
* Generuje przejrzyste **raporty**.
* Może działać w trybie **"white-box"** (model zrozumiały) lub **"black-box"** (maksymalna skuteczność).
* Możliwość działania offline, bez wysyłania danych na serwer.

---

**Instalacja:**

```bash
pip install mljar-supervised
```

---

**Licencja:**
MLJAR AutoML jest udostępniany na licencji MIT, co oznacza, że można go swobodnie używać, modyfikować i rozpowszechniać również w projektach komercyjnych, pod warunkiem zachowania informacji o autorze i licencji.

---

**Podstawowe użycie:**

```python
from supervised.automl import AutoML
import pandas as pd

# Wczytanie danych
df = pd.read_csv("dane.csv")
X = df.drop("target", axis=1)
y = df["target"]

# Inicjalizacja MLJAR AutoML
automl = AutoML(mode="Compete", total_time_limit=300)

# Trening modeli
automl.fit(X, y)

# Predykcja
y_pred = automl.predict(X)
```

---

**Tryby działania:**

* `"Explain"` – szybkie modele, łatwe do interpretacji.
* `"Perform"` – dobre wyniki, rozsądny czas.
* `"Compete"` – maksymalna dokładność, dłuższy czas treningu.
* `"Optuna"` – hiperoptymalizacja z wykorzystaniem biblioteki Optuna.

---

**Wyniki i raporty:**

Po zakończeniu treningu tworzy się folder np. `MLJAR_AutoML/`, który zawiera:

* Raport z wynikami modeli (w nowej wersji zmiast HTML to plik .md, czyli plik tekstowy w formacie Markdown).
* Wykresy ważności cech.
* Szczegóły metryk, konfiguracji i kodu.

## Przygotowanie danych

<div style="text-align: center;"><img src=".//Images//Bank.png" alt="Bank" width="400" height="120" style="margin: 10px; "/></div>

Wykorzystamy zbiór danych z UCI, opisujący kampanię marketingową banku, w której klientom oferowano lokatę terminową. Zmienna docelowa to „yes”, jeśli klient zgodził się na lokatę, i „no”, jeśli nie.

Źródło: https://archive.ics.uci.edu/dataset/222/bank+marketing

*Szerszy opis można znaleźć w materiałach ze spotkania o **Strojeniu modeli**.*

# Przykładowy kod

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from supervised.automl import AutoML
from sklearn.metrics import classification_report

In [2]:
# Wczytanie danych z pliku CSV
df = pd.read_csv('./Data/bank-additional-full.csv', sep=';')

# Usunięcie kolumny 'duration', której nie powinno się używać (bo jest znana tylko po kampanii)
df = df.drop(columns='duration')

# Konwersja kolumny celu 'y' z 'yes'/'no' na 1/0
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# Sprawdzenie kolumn i typów danych (opcjonalnie)
print(df.dtypes)
print(df['y'].value_counts())

# Podział na cechy i etykietę
X = df.drop(columns='y')
y = df['y']

age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                   int64
dtype: object
y
0    36548
1     4640
Name: count, dtype: int64


In [3]:
# Podział danych na treningowe i testowe
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [4]:
# Inicjalizacja MLJAR AutoML
automl = AutoML(
    mode="Compete",
    total_time_limit=60,  # czas może być dłuższy ze względu na większy zbiór
    algorithms=["Xgboost", "LightGBM", "Random Forest", "Extra Trees"],
    eval_metric="logloss",
    validation_strategy={"validation_type": "split", "train_ratio": 0.75},
    explain_level=2
)

# Trening
automl.fit(X_train, y_train)

# Predykcja
y_pred = automl.predict(X_test)

# Raport wyników
print(classification_report(y_test, y_pred))

Disable stacking for split validation
AutoML directory: AutoML_1
The task is binary_classification with evaluation metric logloss
AutoML will use algorithms: ['Xgboost', 'LightGBM', 'Random Forest', 'Extra Trees']
AutoML will ensemble available models
AutoML steps: ['simple_algorithms', 'default_algorithms', 'not_so_random', 'mix_encoding', 'golden_features', 'kmeans_features', 'insert_random_feature', 'features_selection', 'hill_climbing_1', 'hill_climbing_2', 'ensemble']
Skip simple_algorithms because no parameters were generated.
* Step default_algorithms will try to check up to 4 models
1_Default_LightGBM logloss 0.265126 trained in 16.96 seconds
* Step not_so_random will try to check up to 36 models
11_LightGBM logloss 0.26512 trained in 5.39 seconds
2_Xgboost logloss 0.266792 trained in 6.52 seconds
20_RandomForest logloss 0.270858 trained in 11.94 seconds
Skip mix_encoding because no parameters were generated.
Skip golden_features because no parameters were generated.
'score' Tr

Wygenerowany raport znajdziesz w katalogu MLJAR_AutoML (plik .md to zwykły plik tekstowy w formacie Markdown).

# Predykcja prawdopodobieństw

In [6]:
from sklearn.metrics import roc_auc_score

In [7]:
y_proba = automl.predict_proba(X_test)[:, 1]  # prawdopodobieństwo klasy 1
roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC AUC: {roc_auc:.4f}")

ROC AUC: 0.8139


# Raport

In [9]:
print("Ścieżka do wyników MLJAR:", automl._results_path)

Ścieżka do wyników MLJAR: AutoML_1


In [12]:
automl.report()